# Servicing foundation — what we have once a portfolio is acquired

**22 query files** under [work/servicing/](../work/servicing/),
each a grain-checked view of one part of the servicing picture. Together they
query **29 of the 32 objects** in `PROD` (see *Coverage* at the bottom). This
notebook runs **ten of them** — the loan and borrower spine, the money view
that foots, all three forecasts, both restatement views, FX conversion and
acquisition economics. The rest are the same shape and run the same way.

Naming: `ref_` = a reference list, one row per thing. `base_` = the lowest
grain available. `monthly_` = one row per month. `agg_` = a rollup built on
the base layer. Build new `agg_` views on the base layer rather than going
back to the raw facts.

### Reference — one row per thing

| File | Grain — one row per | Answers |
|---|---|---|
| `ref_loan` | loan, at a month-end | What did we buy? Every dimension on one line per loan. |
| `ref_borrower` | borrower | Who are the customers behind the loans? |
| `ref_portfolio` | portfolio | Acquisition economics, revaluations, current size. |
| `ref_calendar` | month-end | Which months actually have data, per fact. |
| `ref_fx_rate` | currency pair × month | Convert money so totals across currencies are valid. |

### Base — lowest grain there is

| File | Grain — one row per | Answers |
|---|---|---|
| `base_collection` | collection transaction | Every individual payment and reversal. |
| `base_loan_carrying` | loan × month-end | What is each loan worth on our books? |
| `base_data_quality` | check | What is missing, wrong, or absent. |

### Monthly — one row per month

| File | Grain — one row per | Answers |
|---|---|---|
| `monthly_loan_performance` | loan × month | How is each loan collecting and amortising? |
| `monthly_borrower_collections` | borrower × month | How does each customer pay? |
| `monthly_portfolio_performance` | portfolio × month × currency | The money view that foots. |
| `monthly_portfolio_forecast` | portfolio × month × forecast type | What do we expect to collect? |
| `monthly_flash_report` | portfolio × month | Actual vs forecast, as the flash report publishes it. |
| `monthly_portfolio_restatement` | portfolio × month × version | Has a reported month changed since we first reported it? |
| `monthly_forecast_restatement` | portfolio × month × run | Are we revising expectations up or down? |

### The collections chain — added 2026-09-11

Six files forming one dbt-shaped chain, so it lifts into `brocc-data` as
models without rework. Each level is the level below grouped by a different
key — no re-derivation, no second read of the fact — so they foot exactly.

| File | Grain — one row per | Answers |
|---|---|---|
| `base_case` | loan × month-end | The case table: who, where, what it owes, every dimension joined. |
| `base_case_collection` | loan × day × deal | What each loan paid, on the day it paid, local and EUR. |
| `agg_portfolio_collection_monthly` | portfolio × month × currency | Collections per portfolio per month. |
| `agg_deal_collection_monthly` | deal × month × currency | The same, per deal. |
| `agg_vehicle_collection_monthly` | SPV × month × currency | The same, per vehicle (`DIM_LEGAL_BUYER`). |
| `agg_servicer_collection_monthly` | servicer × month × currency | The same, per servicer. |
| `base_collection_reconciliation` | check | Re-proves all of the above foot, one month, 14 rows. |

Every `agg_` file carries **two** collection figures per row:
`COLLECTIONS_REPORTED_EUR` (the portfolio fact — the headline, all 250
portfolios, every servicer; **chart this**) and `COLLECTIONS_CASE_EUR` (the
loan-level sum — drillable, and structurally incomplete), plus
`COVERAGE_PCT` / `HAS_CASE_DETAIL`.

⚠ **The loan feed is 60.18% of reported collections** (July 2026:
EUR 24,842,484.52 vs EUR 41,282,117.95) across 138 of 250 portfolios — and
all of it is Intrum. A "by servicer" chart built on the case column shows
Intrum at 100% and deletes Gescobro (EUR 4.85m) and Cerved (EUR 2.13m)
without erroring. Coverage is *data completeness*, never collection
efficiency.

**Read `base_data_quality` first.** Several checks come back BLOCKER, and they
change how everything else must be read.

### Three things to know before quoting a number

1. **Restatement is the norm, not the exception.** 95.7% of portfolio-months
   have more than one version (up to 15). A month-on-month movement read off
   a single vintage is partly restatement, not performance — use
   `monthly_portfolio_restatement` to size it.
2. **`COLLECTION_BK` is not a row key.** 4.1M keys are shared across 47.8M
   rows (36% of the collection fact, EUR 10.6bn), and none of those sharing
   rows are identical — they are different payments. Deduping on it deletes
   real money.
3. **Never roll collections up from the loan level and call it the book's
   collections.** About half the portfolios report no loan detail at all —
   they arrive as portfolio-level aggregates. See the collections chain above.
4. **The flash report and `monthly_portfolio_performance` will disagree** for
   multi-currency portfolios. The mart collapses currency with `any_value()`,
   so its label can be arbitrary while sums mix currencies; the
   `monthly_` file keeps currency in the grain. That difference is the
   explanation, not something to reconcile.

Every `.sql` uses Snowflake session variables (`set month_end = ...`), so the
same file runs unchanged in the Snowflake VS Code extension (Run All), in the
terminal (`mise run sql -- <file>`), and here.

In [ ]:
import datalake
import pandas as pd

# Money and ratios to 5 dp. Display formatting only — the DataFrame keeps
# full precision, so never use .round() to tidy output.
pd.set_option("display.float_format", lambda v: f"{v:,.5f}".rstrip("0").rstrip("."))
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# The last COMPLETE month. 2026-08-31 exists but only ~3% of portfolios have
# loaded — see the first data-quality check.
MONTH_END = "2026-07-31"

datalake.session_info()

## 0 — Data quality first

Run this before trusting anything below. It measures live rather than
repeating what was true when the files were written.

In [ ]:
dq = datalake.q("servicing/prod/base_data_quality", month_end=MONTH_END)
for _, r in dq.iterrows():
    print(f"[{r.SEVERITY:7}] {r.CHECK_NAME}")
    print(f"           {r.MEASURED}")
    print(f"           -> {r.WHAT_TO_DO}\n")

### The four blockers, in plain terms

1. **The newest month is a load lag, not a business event.** Only 7 of 250
   portfolios have August data. Use July as the last complete month.
2. **112 of 250 portfolios have no loan-level detail at all.** They report as
   portfolio aggregates only (the Intrum PAM feed). Any loan-level analysis
   silently omits them.
3. **Where both exist they disagree** — loan level sees ~77% of the money.
   Say which basis a collections number is on.
4. **Servicer is unattributable at portfolio level** —
   `REPORTING_SYSTEM_SK` is 100% NULL there. Use `DEAL_SOURCE_NAME`, or get
   servicer at loan level.

## 1 — Loan master: one row per loan

The spine. Pick a portfolio and pull every loan on book with all dimensions
resolved.

Set `portfolio_filter` to a LIKE pattern. `'%'` is all 250 portfolios and
~21.8M loans — far too much for pandas, so keep it narrow.

In [ ]:
PORTFOLIO = "LOK-DE24016-U01/2512"      # a German Intrum portfolio, ~8k loans:
                                        # small enough that nothing here truncates

loans = datalake.q(
    "servicing/prod/ref_loan",
    month_end=MONTH_END,
    portfolio_filter=PORTFOLIO,
    row_limit=100000,
)

print(f"{len(loans):,} loans | {loans.CASE_BK.nunique():,} distinct CASE_BK")
loans[[
    "CASE_BK", "PORTFOLIO_BK", "SERVICER_NAME", "SECURITY_CLASS_NAME",
    "CURRENCY_ID", "OUTSTANDING_TOTAL", "COLLECTED_LIFETIME",
    "N_BORROWERS", "MONTHS_SINCE_LAST_PAYMENT",
]].head(10)

In [ ]:
# ⚠ Always check the cap. A truncated result still sums happily.
LIMIT = 100000
if len(loans) == LIMIT:
    print(f"TRUNCATED at {LIMIT:,} rows — totals below are WRONG. Narrow the filter.")
else:
    print(f"Complete: {len(loans):,} rows, under the {LIMIT:,} cap.")

In [ ]:
# What is actually populated? The header of each .sql lists measured null
# rates for the whole book; this is the same check for YOUR slice.
null_pct = (loans.isna().mean() * 100).sort_values(ascending=False)
null_pct[null_pct > 0].to_frame("pct_null").head(20)

## 2 — How each loan is collecting, month by month

One row per loan per month-end: opening balance, collections, closing
balance, and the movement collections do *not* explain.

In [ ]:
loan_months = datalake.q(
    "servicing/prod/monthly_loan_performance",
    month_end=MONTH_END,
    months_back=12,
    portfolio_filter=PORTFOLIO,
    row_limit=500000,
)

print(f"{len(loan_months):,} rows | duplicate (loan, month) pairs: "
      f"{loan_months.duplicated(['CASE_SK', 'MONTH_END']).sum()}")

by_month = loan_months.groupby("MONTH_END").agg(
    loans=("CASE_BK", "nunique"),
    paying_loans=("PAID_THIS_MONTH", "sum"),
    collected=("COLLECTED", "sum"),
    closing_balance=("CLOSING_BALANCE", "sum"),
    unexplained=("UNEXPLAINED_MOVEMENT", "sum"),
)
by_month["pct_paying"] = 100 * by_month.paying_loans / by_month.loans
by_month

`UNEXPLAINED_MOVEMENT` is balance movement that collections do not account
for — interest accrual, fees, write-offs, FX, servicer restatements. It is
exposed rather than hidden so you can size it. A large value means the
balance series is being driven by something other than recovery.

`OPENING_BALANCE` is NULL in each loan's first month of the window. That is
a `lag()` edge, not missing data.

## 3 — The customers behind the loans

One row per borrower.

**The balances here do not foot.** A collection is recorded against a *loan*,
and 8.2M of 22.5M loans have two or more borrowers with no split available —
so a joint loan's balance appears in full against each borrower. Per borrower
the figure is right; summed across borrowers it double counts.

In [ ]:
borrowers = datalake.q(
    "servicing/prod/ref_borrower",
    month_end=MONTH_END,
    portfolio_filter=PORTFOLIO,
    row_limit=100000,
)

print(f"{len(borrowers):,} borrowers | "
      f"{borrowers.IS_MULTI_LOAN.sum():,} on more than one loan | "
      f"{borrowers.NEVER_PAID.sum():,} have never paid")

# Proof of the double count: the sum across borrowers should EXCEED the
# portfolio's real balance, because joint loans are counted once per borrower.
print(f"\nsum of borrower balances : {borrowers.TOTAL_BALANCE_UNAPPORTIONED.sum():>20,.2f}")
print(f"true loan-level balance  : {loans.OUTSTANDING_TOTAL.sum():>20,.2f}")

borrowers[[
    "BORROWER_ID", "LEGAL_EBA_TYPE", "GENDER", "LOAN_COUNTRY_ID", "N_LOANS",
    "TOTAL_BALANCE_UNAPPORTIONED", "COLLECTED_LIFETIME_UNAPPORTIONED", "NEVER_PAID",
]].head(10)

`DIM_BORROWER` carries **no name, address, phone or email** — only
pseudonymous ids and a few attributes, most of them sparse (VAT 99.6% null,
credit rating 84.8% null). This is a segmentation sheet, not a contactable
customer list. Contact data for a servicing workflow is not in the warehouse.

## 4 — How customers pay, month by month

One row per borrower per month **in which they paid**. Months with no payment
are absent, not zero.

In [ ]:
borrower_months = datalake.q(
    "servicing/prod/monthly_borrower_collections",
    month_end=MONTH_END,
    months_back=24,
    portfolio_filter=PORTFOLIO,
    row_limit=500000,
)

affected = (borrower_months.N_JOINT_LOANS > 0).sum()
print(f"{len(borrower_months):,} borrower-months | "
      f"{affected:,} affected by joint-loan double counting")
print(f"collected (may double count) : {borrower_months.COLLECTED.sum():>18,.2f}")
print(f"sole-borrower loans only     : "
      f"{borrower_months.COLLECTED_SOLE_BORROWER_ONLY.sum():>18,.2f}")

# Payment behaviour: how many months out of the window did each customer pay?
behaviour = borrower_months.groupby("BORROWER_ID").agg(
    paying_months=("MONTH_END", "nunique"),
    total_paid=("COLLECTED", "sum"),
)
behaviour.paying_months.value_counts().sort_index().to_frame("borrowers").head(15)

## 5 — Portfolio performance: the view that foots

Never goes through the case or borrower grain, so it covers all 250
portfolios including the 112 with no loan-level detail. Grain is portfolio ×
month × **currency** — a portfolio can span up to 5 currencies, so a
portfolio-only total would add EUR to DKK.

In [ ]:
portfolios = datalake.q(
    "servicing/prod/monthly_portfolio_performance",
    month_end=MONTH_END,
    months_back=12,
    portfolio_filter="%",          # all portfolios: this table is small
    row_limit=100000,
)

print(f"{len(portfolios):,} rows | "
      f"{portfolios.PORTFOLIO_BK.nunique()} portfolios | "
      f"{portfolios.CURRENCY_ID.nunique()} currencies")

# Coverage of the loan-level feed, per month. This is the single most
# important number to know before quoting any loan-level total.
cov = portfolios.groupby("PERIOD_END_DATE").agg(
    portfolios=("PORTFOLIO_BK", "nunique"),
    with_loan_detail=("HAS_LOAN_DETAIL", "sum"),
    portfolio_level=("GROSS_CASH_RECOVERIES", "sum"),
    loan_level=("CASE_LEVEL_COLLECTIONS", "sum"),
)
cov["loan_level_coverage_pct"] = 100 * cov.loan_level / cov.portfolio_level
cov

The `portfolio_level` and `loan_level` columns above mix currencies, so treat
them as a coverage *ratio* rather than a total. For a real total, filter to
one currency or convert via `COMMON.FCT_EXCHANGE_RATE_MONTHLY`.

In [ ]:
# Single-currency view, so the money is genuinely addable.
eur = portfolios[portfolios.CURRENCY_ID == "EUR"]

eur.groupby("PERIOD_END_DATE").agg(
    portfolios=("PORTFOLIO_BK", "nunique"),
    gross_recoveries=("GROSS_CASH_RECOVERIES", "sum"),
    servicing_cost=("TOTAL_SERVICING_COST", "sum"),
    net_recoveries=("NET_CASH_RECOVERIES", "sum"),
    carrying_closing=("CARRYING_CLOSING", "sum"),
)

`COMMISSION`, `PASSTHROUGH_COST`, `UNREMITTED_COLLECTIONS` and
`NET_CASH_RECOVERIES` are **Intrum-only**. For every other servicer they are
zero — not because servicing is free, but because the data is not supplied.
So `NET_CASH_RECOVERIES` equals `GROSS_CASH_RECOVERIES` elsewhere, and a
cross-servicer net-margin comparison is invalid. `IS_NET_MEANINGFUL` flags
the rows where the split is real.

In [ ]:
portfolios.groupby(["DEAL_SOURCE_NAME", "IS_NET_MEANINGFUL"]).agg(
    portfolios=("PORTFOLIO_BK", "nunique"),
    gross=("GROSS_CASH_RECOVERIES", "sum"),
    servicing_cost=("TOTAL_SERVICING_COST", "sum"),
)

## 6 — What we expect to collect: three forecasts

`monthly_portfolio_forecast` stacks all three forecasts the warehouse holds
into one shape. **They are different questions. Do not average them.**

| Forecast | The question it answers |
|---|---|
| `UNDERWRITING` | What we assumed when we *bought* the portfolio — the original investment case |
| `ACCOUNTING` | The *current* live forecast, the one carrying amount is built from |
| `SERVICER` | What the servicer themselves projects — the finest grain, and the only one carrying `SEGMENT_SK` |

Underwriting vs accounting is *"are we on plan"*. Accounting vs actual is
*"are we on the current forecast"*. Servicer vs accounting is *"do we and the
servicer agree"*.

**The date axis is `PERIOD_END_DATE`** — the month the money is expected in.
It means exactly what it means in `monthly_portfolio_performance`, where it is
built as `last_day(collection_date)`, and that shared meaning is what makes
actual-vs-forecast possible at all. `REPORTING_PERIOD_END_DATE` is the
*vintage*: the month the forecast was generated. It is provenance, not a time
axis — see section 8 if that is the question you actually have.

In [ ]:
forecast = datalake.q(
    "servicing/prod/monthly_portfolio_forecast",
    months_back=24,
    months_ahead=24,
    portfolio_filter="%",
    row_limit=200000,
)

print(f"{len(forecast):,} rows | {forecast.PORTFOLIO_BK.nunique()} portfolios")

forecast.groupby("FORECAST_TYPE").agg(
    rows=("PORTFOLIO_BK", "size"),
    portfolios=("PORTFOLIO_BK", "nunique"),
    first_month=("PERIOD_END_DATE", "min"),
    last_month=("PERIOD_END_DATE", "max"),
    vintages=("REPORTING_PERIOD_END_DATE", "nunique"),
)

In [ ]:
# The two guards the .sql already applied, made visible rather than silent.
#
#   N_VERSIONS_AVAILABLE > 1 — this portfolio-month had later revisions in the
#   servicer forecast and only the newest was kept. Summing all versions would
#   add a forecast to its own replacement.
#
#   N_CURRENCIES > 1 — the roll-up mixed currencies, so the money on that row
#   is not addable as it stands. Converted properly in section 7.
revised = (forecast.N_VERSIONS_AVAILABLE > 1).sum()
mixed   = (forecast.N_CURRENCIES > 1).sum()
print(f"portfolio-months where revisions were dropped : {revised:,}")
print(f"rows mixing currencies (not addable as-is)    : {mixed:,}")

# Unknown-portfolio rows are labelled, not hidden — see the .sql header.
unknown = forecast[forecast.PORTFOLIO_BK.str.startswith("(unknown portfolio")]
print(f"rows on the Unknown portfolio sentinel        : {len(unknown):,}")

In [ ]:
# ⚠ NEVER put a forecast total and an actual total in the same column without
# filtering on IS_ACTUAL_PERIOD. These forecasts run to 2041-06-30 and almost
# all of that is projection.
#
# ⚠ This view MIXES CURRENCIES, so read it as the shape of the curve, not as a
# total. Section 7 converts to EUR and gives a number you can quote.
curve = forecast.pivot_table(
    index="PERIOD_END_DATE",
    columns="FORECAST_TYPE",
    values="GROSS_COLLECTION_FORECAST",
    aggfunc="sum",
)
curve["is_actual_period"] = curve.index <= pd.Timestamp(MONTH_END)
curve.tail(30)

## 7 — Actual vs expected, in one currency

The headline number, end to end. This needs FX, and FX here has two traps that
between them can put a 122x error on a chart:

1. **The rate is already the multiplier.** Despite the column names, the values
   are *EUR per unit of the foreign currency* — one GBP is 1.17 EUR, one SEK is
   0.09 EUR. **Multiply. Do not invert.** Inverting values SEK 32.2m of July
   collections at EUR 356m instead of EUR 2.9m, and nothing errors.
2. **`RATE_TYPE = 'FIXED'` converts nothing.** It exists only as the identity
   self-pair. Every real cross rate is `PMI`.

`ref_fx_rate.sql` unions the identity row in at 1.0, so a NULL rate means
*genuinely unconvertible* rather than "already in the target currency" — which
would otherwise silently drop the largest block of all.

Collections are a **flow**, so use `RATE_TO_BASE_MONTHLY_AVG`. For balances and
carrying amounts use `RATE_TO_BASE_ULTIMO` instead.

In [ ]:
fx = datalake.q(
    "servicing/prod/ref_fx_rate",
    rate_type="PMI",
    to_currency="EUR",
    date_from="2018-01-31",
    date_to="2026-12-31",
)

rates = fx[["MONTH_END", "FROM_CURRENCY", "RATE_TO_BASE_MONTHLY_AVG"]]
print(f"{len(fx):,} rate rows | {fx.FROM_CURRENCY.nunique()} currencies "
      f"| identity row present: {(fx.FROM_CURRENCY == 'EUR').any()}")

In [ ]:
# ---- actuals, converted to EUR -------------------------------------------
# `portfolios` comes from section 5 and is grained portfolio x month x currency,
# so CURRENCY_ID is a single code and joins to the rate table directly.
act = portfolios.merge(
    rates,
    left_on=["PERIOD_END_DATE", "CURRENCY_ID"],
    right_on=["MONTH_END", "FROM_CURRENCY"],
    how="left",
)
act["GROSS_EUR"] = act.GROSS_CASH_RECOVERIES * act.RATE_TO_BASE_MONTHLY_AVG

unconvertible = act.RATE_TO_BASE_MONTHLY_AVG.isna().sum()
print(f"actual rows with no rate (excluded from the EUR total): {unconvertible:,}")
if unconvertible:
    print(act[act.RATE_TO_BASE_MONTHLY_AVG.isna()].CURRENCY_ID.value_counts())

In [ ]:
# ---- the accounting forecast, converted to EUR ----------------------------
# ⚠ monthly_portfolio_forecast aggregates currency AWAY — it carries CURRENCIES
# (a list) and N_CURRENCIES, not a single CURRENCY_ID. Only single-currency rows
# can be converted, so the mixed ones are EXCLUDED AND COUNTED rather than
# quietly dropped. For a currency-safe forecast at row grain, use
# monthly_forecast_restatement, which does carry CURRENCY_ID.
fc = forecast[
    (forecast.FORECAST_TYPE == "ACCOUNTING")
    & forecast.IS_ACTUAL_PERIOD
].copy()

excluded = (fc.N_CURRENCIES > 1).sum()
print(f"forecast rows excluded as multi-currency: {excluded:,} of {len(fc):,}")

fc = fc[fc.N_CURRENCIES == 1].copy()
fc["CURRENCY_ID"] = fc.CURRENCIES          # single-valued when N_CURRENCIES == 1

fc = fc.merge(
    rates,
    left_on=["PERIOD_END_DATE", "CURRENCY_ID"],
    right_on=["MONTH_END", "FROM_CURRENCY"],
    how="left",
)
fc["FORECAST_EUR"] = fc.GROSS_COLLECTION_FORECAST * fc.RATE_TO_BASE_MONTHLY_AVG
print(f"forecast rows with no rate: {fc.RATE_TO_BASE_MONTHLY_AVG.isna().sum():,}")

In [ ]:
# ---- the comparison -------------------------------------------------------
variance = pd.DataFrame({
    "actual_eur":   act.groupby("PERIOD_END_DATE").GROSS_EUR.sum(),
    "expected_eur": fc.groupby("PERIOD_END_DATE").FORECAST_EUR.sum(),
})
variance["variance_eur"] = variance.actual_eur - variance.expected_eur
variance["pct_of_expected"] = 100 * variance.actual_eur / variance.expected_eur
variance

Above 100% means we collected more than the current accounting forecast
expected. Read it against section 5's coverage table: a month where the
loan-level feed has not fully loaded will understate the actual side, and the
newest month is a load lag rather than a business event.

This compares against `ACCOUNTING` — *are we on the current forecast*. Swap
`FORECAST_TYPE` to `UNDERWRITING` for *are we on the original plan*, which is a
different and usually more interesting question for a portfolio past its first
year.

## 8 — Is the number moving? Restatement

**Restatement is the normal state of portfolio actuals here, not an anomaly.**
A month you reported last quarter may not hold the same value today. Two
separate questions, two files:

| Question | File | Version key |
|---|---|---|
| Has a **reported actual** changed? | `monthly_portfolio_restatement` | `SNAPSHOT_VERSION` |
| Are we **revising expectations**? | `monthly_forecast_restatement` | `REPORTING_DATE` |

⚠ The forecast one uses **`REPORTING_DATE`**, not `SNAPSHOT_VERSION` (single-valued
in that table) and not `REPORTING_PERIOD_END_DATE` (79 short of unique, so
grouping on it silently merges runs). Getting this wrong collapses the history
without erroring.

In [ ]:
restated = datalake.q(
    "servicing/prod/monthly_portfolio_restatement",
    month_end=MONTH_END,
    months_back=12,
    portfolio_filter="%",
    row_limit=200000,
)

multi = restated[restated.N_VERSIONS_FOR_MONTH > 1]
print(f"{len(restated):,} rows | "
      f"{multi.PORTFOLIO_BK.nunique()} portfolios restated at least once")

# Size of the restatements that actually moved a number.
moved = restated[restated.RESTATEMENT_VS_PREVIOUS.notna()
                 & (restated.RESTATEMENT_VS_PREVIOUS != 0)]
moved.groupby("PERIOD_END_DATE").agg(
    portfolios=("PORTFOLIO_BK", "nunique"),
    restatements=("RESTATEMENT_VS_PREVIOUS", "size"),
    net_change=("RESTATEMENT_VS_PREVIOUS", "sum"),
    largest_up=("RESTATEMENT_VS_PREVIOUS", "max"),
    largest_down=("RESTATEMENT_VS_PREVIOUS", "min"),
)

In [ ]:
fc_restated = datalake.q(
    "servicing/prod/monthly_forecast_restatement",
    months_back=12,
    months_ahead=12,
    portfolio_filter="%",
    row_limit=200000,
)

revised = fc_restated[fc_restated.REVISION_VS_PREVIOUS.notna()
                      & (fc_restated.REVISION_VS_PREVIOUS != 0)]
print(f"{len(fc_restated):,} rows | {fc_restated.REPORTING_DATE.nunique()} distinct runs")

# ⚠ CURRENCY_ID is real here, so never sum across it without converting.
revised.groupby(["PERIOD_END_DATE", "CURRENCY_ID"]).agg(
    portfolios=("PORTFOLIO_BK", "nunique"),
    net_revision=("REVISION_VS_PREVIOUS", "sum"),
).head(30)

## 9 — Acquisition economics: what we paid and what came back

`ref_portfolio` is the one-row-per-portfolio sheet: capex, GBV, purchase price,
underwriting IRR, and `MONTHS_SINCE_CAPEX`.

⚠ **Acquisition vintage only.** True loan origination vintage is not available —
`ORIGINATION_DATE` is 95.4% null — so any "vintage curve" here is keyed on when
*we bought* the portfolio, not when the loans were written. Worth stating
explicitly before anyone reads the chart the other way.

In [ ]:
port = datalake.q(
    "servicing/prod/ref_portfolio",
    month_end=MONTH_END,
    portfolio_filter="%",
)

print(f"{len(port):,} portfolios | "
      f"{(port.N_CURRENCIES > 1).sum()} spanning more than one currency")

# Single-currency EUR portfolios, so price and GBV are genuinely addable.
eur_port = port[(port.N_CURRENCIES == 1) & (port.CURRENCIES == "EUR")]
print(f"{len(eur_port):,} single-currency EUR portfolios")

eur_port.groupby("DEAL_SOURCE_NAME").agg(
    portfolios=("PORTFOLIO_BK", "nunique"),
    gbv=("GBV", "sum"),
    price_gross=("PRICE_GROSS", "sum"),
    capex=("CAPEX_AMOUNT", "sum"),
    mean_price_pct=("PRICE_PERCENT", "mean"),
    mean_uw_irr=("UW_UNLEVERED_IRR", "mean"),
    mean_months_held=("MONTHS_SINCE_CAPEX", "mean"),
)

## What a dashboard built on this *cannot* answer

Tested against the warehouse on 2026-09-09, not assumed. These are real
absences in the data, not gaps in the queries — no amount of SQL fixes them,
and each is a conversation with the data team.

| Question | Why not |
|---|---|
| **Cost to serve, across servicers** | **Intrum only.** Commission, passthrough and net recoveries are *zero* for every other servicer — not free, just not supplied. `IS_NET_MEANINGFUL` flags it. Any cross-servicer net-margin comparison is invalid. |
| **Operational KPIs** — calls, letters, promises to pay, legal stage, FTE | **Not in the warehouse at all.** This answers *"how much came in and how does that compare to plan"* — a financial view. It cannot answer *"how well is the collections operation running"*. |
| **Cure and re-default rates** | `FCT_CASE_FORBEARANCE` is one row per case with no end date, so duration cannot be derived. |
| **Origination vintage curves** | `ORIGINATION_DATE` 95.4% null. Acquisition vintage only. |
| **Per-borrower split of a joint loan** | `BRG_CASE_BORROWER` has no share or primary-borrower flag. 8.2m of 22.5m loans have two or more borrowers. |
| **Contactable customer list** | `DIM_BORROWER` holds no name, address, phone or email. |
| **Portfolio-level servicer attribution** | `FCT_PORTFOLIO_SNAPSHOT_MONTHLY.REPORTING_SYSTEM_SK` is 100% NULL. Use `DEAL_SOURCE_NAME`. |

The second row is the one most likely to be assumed present by a stakeholder.
Worth raising before it is promised.

## Where to go next

Sections 0-9 are the foundation, not the analysis. What they set up:

- **Collection curves by acquisition vintage** — section 7 gives actual vs
  expected per portfolio-month in EUR, and `MONTHS_SINCE_CAPEX` from section 9
  gives the age. Group one by the other.
- **Which portfolios are missing plan, and by how much** — section 7's
  `variance` table, broken out by `DEAL_SOURCE_NAME` or `PORTFOLIO_BK` rather
  than rolled to a single month.
- **Underwriting vs accounting drift** — rerun section 7 with
  `FORECAST_TYPE == "UNDERWRITING"` to ask "are we on the original plan"
  instead of "are we on the current forecast".
- **Do we and the servicer agree?** — the `SERVICER` forecast against
  `ACCOUNTING`, same axis, from the `curve` table in section 6.
- **Servicer performance** — only sound at loan level, via `ref_loan` /
  `monthly_loan_performance` `SERVICER_NAME`. At portfolio level use
  `DEAL_SOURCE_NAME`; `REPORTING_SYSTEM_SK` is 100% NULL.
- **Payer segmentation** — section 4 gives paying-month counts per borrower.

The five files this notebook does not run — `base_collection`,
`base_loan_carrying`, `base_reconciliation`, `ref_calendar`,
`monthly_flash_report` — take the same arguments and return the same shape.
`monthly_flash_report` is worth knowing about specifically: it reproduces what
the flash report publishes, and **will disagree** with section 5 for
multi-currency portfolios, for the reason given in the header.

See *What a dashboard built on this cannot answer*, above, before promising
anything operational.

---

## Coverage — what is deliberately NOT here

`PROD` holds 32 objects across `NPL_COMMON` (26) and `COMMON` (6). The 15
files query **29**. The three that are not, and why:

| Object | Why not |
|---|---|
| `NPL_COMMON.DIM_FORECAST_SOURCE` | **Unusable.** `FORECAST_SOURCE_SK` appears in exactly one column schema-wide — its own. It joins to nothing, so there is no way to use it. |
| `COMMON.DIM_DATE` | Daily calendar, but every fact here is month-end only — `FCT_CASE_SNAPSHOT_DAILY` holds 147 dates and all of them are month-ends. `ref_calendar` uses `DIM_MONTH`, the matching grain. |
| `COMMON.FCT_EXCHANGE_RATE_DAILY` | Same reason: no daily-grained fact to convert. `ref_fx_rate` uses the monthly table, and picks `ULTIMO_VALUE` for balances vs `MONTHLY_AVG` for flows. |

Verified 2026-09-09 by compiling every file against the warehouse and
checking each file's stated grain against its actual output.